In [5]:
from importlib.metadata import version

print(f"torch:  {version("torch")}")
print(f"transformers:  {version("transformers")}")
print(f"bitsandbytes:  {version("bitsandbytes")}")
print(f"sklearn:  {version("scikit-learn")}")

torch:  2.7.0+cu128
transformers:  4.56.1
bitsandbytes:  0.48.1
sklearn:  1.7.2


In [ ]:
import json
import torch
import random
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from bitsandbytes.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 设置随机种子

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# 1. 数据预处理和dataset

In [6]:
# 这里不用去除停用词，因为特殊字符，标点，表情符号都是判断是否是低质量文本有用的特征
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        # 使用BERT tokenizer编码文本
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }



# 2. 创建示例数据

In [7]:
def create_data(path):
    texts = []
    labels = []
    with open(path) as fd:
        for line in fd:
            info = json.loads(line)
            texts.append(info["text"])
            label = 1 if info["spam"] == "Y" else 0
            labels.append(label)
    return texts, labels

# 3. 训练函数

In [8]:
def train_model(model, train_loader, val_loader, device, save_path, epochs=3, learning_rate=2e-5):
    """训练BERT分类模型"""
    
    # 优化器和学习率调度器
    optimizer = AdamW(model.parameters(), lr=learning_rate)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=total_steps
    )
    
    # 损失函数
    criterion = nn.CrossEntropyLoss().to(device)
    
    best_accuracy = 0
    training_stats = []
    
    for epoch in range(epochs):
        print(f'\nEpoch {epoch + 1}/{epochs}')
        print('-' * 50)
        
        # 训练阶段
        model.train()
        total_train_loss = 0
        train_correct = 0
        train_total = 0
        
        for batch_idx, batch in enumerate(train_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            # 前向传播
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            logits = outputs.logits
            
            # 反向传播
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # 梯度的裁剪
            optimizer.step()
            scheduler.step()
            
            # 统计信息
            total_train_loss += loss.item()
            _, predicted = torch.max(logits, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
            
            if batch_idx % 10 == 0:
                print(f'Batch {batch_idx}/{len(train_loader)}, Loss: {loss.item():.4f}')
        
        # 验证阶段
        model.eval()
        total_val_loss = 0
        val_correct = 0
        val_total = 0
        all_predictions = []
        all_labels = []
        
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)
                
                outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss
                logits = outputs.logits
                
                total_val_loss += loss.item()
                _, predicted = torch.max(logits, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
                
                all_predictions.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        # 计算准确率
        train_accuracy = 100 * train_correct / train_total
        val_accuracy = 100 * val_correct / val_total
        
        avg_train_loss = total_train_loss / len(train_loader)
        avg_val_loss = total_val_loss / len(val_loader)
        
        print(f'Train Loss: {avg_train_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%')
        print(f'Val Loss: {avg_val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%')
        
        # 保存最佳模型
        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy
            torch.save(model.state_dict(), save_path)
            print(f'保存最佳模型，验证准确率: {val_accuracy:.2f}%')
        
        # 记录训练统计
        training_stats.append({
            'epoch': epoch + 1,
            'train_loss': avg_train_loss,
            'train_accuracy': train_accuracy,
            'val_loss': avg_val_loss,
            'val_accuracy': val_accuracy
        })
    
    return training_stats


# 4. 测试函数

In [9]:
def test_model(model, test_loader, device):
    """测试模型性能"""
    model.eval()
    all_predictions = []
    all_labels = []
    all_probabilities = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, total=len(test_loader), desc="Test Phase"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            probabilities = torch.softmax(logits, dim=1)
            
            _, predicted = torch.max(logits, 1)
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())
    
    # 计算评估指标
    accuracy = accuracy_score(all_labels, all_predictions)
    class_report = classification_report(all_labels, all_predictions, target_names=['高质', '低质'])
    conf_matrix = confusion_matrix(all_labels, all_predictions)
    
    print(f'\n测试结果:')
    print(f'准确率: {accuracy:.4f}')
    print(f'\n分类报告:')
    print(class_report)
    print(f'\n混淆矩阵:')
    print(conf_matrix)
    
    return {
        'predictions': all_predictions,
        'labels': all_labels,
        'probabilities': all_probabilities,
        'accuracy': accuracy,
        'classification_report': class_report,
        'confusion_matrix': conf_matrix
    }


In [10]:
def predict_spam(model, tokenizer, text, device, max_length=512):
    """预测batch文本的label"""
    model.eval()
    
    encoding = tokenizer(
        text,
        padding=True, 
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1)
        _, prediction = torch.max(logits, 1)
    
    pred = prediction.detach().cpu().numpy().flatten()# 0代表高质量，1代表低质量
    confidence = probabilities[:, 0].detach().cpu().numpy().flatten() # 高质量的置信度
    return pred, confidence

In [11]:
! head -3 "data/output_quality_full.json"

{"text": "请问深入骨髓地喜欢一个人怎么办我不能确定对方是不是喜欢我，我却想 一定要告诉他你很喜欢他 很爱他!!  虽然不知道你和他现在的关系是什么！但如果真的觉得很喜欢就向他表白啊！！起码你努力过了！  女生主动多少占一点优势的！！呵呵  只愿曾经拥有！  到以后就算感情没现在这么强烈了也不会觉得遗憾啊~！  与其每天那么痛苦的想他 恋他 还不如直接告诉他 ！  不要怕回破坏你们现有的感情！因为如果不告诉他  你可能回后悔一辈子！！  ", "spam": "N"}
{"text": "我登陆诛仙2时总说我账号密码错误，但是我打的是正确的，就算不对我? 被盗号了~我的号在22号那天被盗了，跟你一样情况，link密码与账号错误，我密保都有了呐，邮箱换密码也不行，还被删了号，伤心兼郁闷，呵呵，盗号了。建议跟完美申请把号要回来，或者玩新的号！", "spam": "Y"}
{"text": "斩魔仙者称号怎么得来的 楼主您好，以下为转载：\r\r圣诞前热身 来《生肖传说》做斩魔仙者\r\r　　一年一度的圣诞节快要来临了，大街小巷商户们都在忙着准备12月25日圣诞的来临。而这时候，一些妖魔也正蠢蠢欲动准备作乱。作为生肖世界肩负维护世界和平、拯救全人类的生肖使者，怎么能不有所行动，为了生肖世界的安定而做防范准备？！\r\r　　要让妖魔鬼怪能对你有所心悸，除了自己本身武艺要高强，最好能在妖魔界打出知名度，这样，当你的亲朋好友被妖魔袭击时，只要爆出你的名号，这些妖魔上就会落荒而逃，岂不好哉？那么，“斩魔仙者”这个响亮的称号应该足够能震慑住妖魔，让他们铭记在心了吧！\r\r斩魔仙者的称号\r\r　　而且，这个“斩魔仙者”的称号并不是人人都能得到的。只有成功挑战70级副本中的隐藏BOSS“羽翼仙”的人才能获得此称号！并且前提条件是在12月18日~12月25日之间第一队成功挑战羽翼仙的人才能获此称号！因此，此称号在全服范围内，是绝对不可能超过5个的！\r\r　　要挑战羽翼仙可不是一件容易的事。首先，要在70级副本中打败4个强大的BOSS！在打完副本的第4个BOSS有一定几率获得道具“羽翼真元”，有了羽翼真元后就可以与羽翼仙进行一场战斗。羽翼仙就站在第4个BOSS的旁边，只是没有道具是不能进入战斗的。\r\r羽翼仙\r\r　　在12月18日~12月25日活动期间成功挑战羽翼仙后

# 主函数

In [12]:
# 设置 GPU/CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用设备: {device}')

# 模型参数设置
NUM_LABELS = 2
BATCH_SIZE = 64
MAX_LENGTH = 512
EPOCHS = 3
LEARNING_RATE = 3e-5
PRETRAINED_PATH = 'bert-base-chinese'
CKPT_PATH = "saved_model/best_bert_classifier.pth" 
DATA_PATH = "../data/output_quality_full.json"

# 加载中文BERT tokenizer和模型
tokenizer = BertTokenizer.from_pretrained(PRETRAINED_PATH)
model = BertForSequenceClassification.from_pretrained(
    PRETRAINED_PATH,
    num_labels=NUM_LABELS,  # 二分类
    output_attentions=False,
    output_hidden_states=False
)
model = model.to(device)

使用设备: cuda


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
# tokenizer测试
tokenizer.encode_plus(
            "今年德国队世界杯夺冠了",
            add_special_tokens=True,
            max_length=20,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

{'input_ids': tensor([[ 101,  791, 2399, 2548, 1744, 7339,  686, 4518, 3344, 1932, 1094,  749,
          102,    0,    0,    0,    0,    0,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0]])}

# 准备数据

In [13]:
# 准备数据
texts, labels = create_data(DATA_PATH)

# 分割数据集
train_texts, other_texts, train_labels, other_labels = train_test_split(
    texts, labels, test_size=0.3
)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    other_texts, other_labels, test_size=0.5
)

print(f'训练集大小: {len(train_texts)}')
print(f'验证集大小: {len(val_texts)}')
print(f'测试集大小: {len(test_texts)}')

# 创建数据集和数据加载器
train_dataset = TextClassificationDataset(train_texts, train_labels, tokenizer, MAX_LENGTH)
val_dataset = TextClassificationDataset(val_texts, val_labels, tokenizer, MAX_LENGTH)
test_dataset = TextClassificationDataset(test_texts, test_labels, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

训练集大小: 31480
验证集大小: 6746
测试集大小: 6746


# 训练模型

In [12]:
# 训练模型
print("开始训练模型...")
training_stats = train_model(model, train_loader, val_loader, device, CKPT_PATH, EPOCHS, LEARNING_RATE)
json.dump(training_stats, open("../data/bert_training_stats.json", "w"))

开始训练模型...

Epoch 1/3
--------------------------------------------------
Batch 0/492, Loss: 0.6147
Batch 10/492, Loss: 0.4615
Batch 20/492, Loss: 0.4562
Batch 30/492, Loss: 0.5643
Batch 40/492, Loss: 0.3917
Batch 50/492, Loss: 0.3606
Batch 60/492, Loss: 0.4503
Batch 70/492, Loss: 0.3477
Batch 80/492, Loss: 0.3062
Batch 90/492, Loss: 0.3242
Batch 100/492, Loss: 0.4231
Batch 110/492, Loss: 0.4053
Batch 120/492, Loss: 0.2895
Batch 130/492, Loss: 0.2297
Batch 140/492, Loss: 0.3280
Batch 150/492, Loss: 0.3167
Batch 160/492, Loss: 0.3914
Batch 170/492, Loss: 0.2225
Batch 180/492, Loss: 0.2909
Batch 190/492, Loss: 0.2985
Batch 200/492, Loss: 0.4791
Batch 210/492, Loss: 0.3612
Batch 220/492, Loss: 0.3524
Batch 230/492, Loss: 0.3723
Batch 240/492, Loss: 0.2505
Batch 250/492, Loss: 0.2835
Batch 260/492, Loss: 0.3571
Batch 270/492, Loss: 0.3001
Batch 280/492, Loss: 0.3249
Batch 290/492, Loss: 0.3655
Batch 300/492, Loss: 0.2562
Batch 310/492, Loss: 0.3773
Batch 320/492, Loss: 0.2566
Batch 330/492, 

# 测试模型

In [14]:
set_seed(42)

# 加载最佳模型进行测试
print("\n加载最佳模型进行测试...")
model.load_state_dict(torch.load(CKPT_PATH))

# 测试模型
test_results = test_model(model, test_loader, device)


加载最佳模型进行测试...


Test Phase: 100%|██████████| 106/106 [00:44<00:00,  2.36it/s]


测试结果:
准确率: 0.8658

分类报告:
              precision    recall  f1-score   support

          高质       0.87      0.95      0.91      4872
          低质       0.83      0.65      0.73      1874

    accuracy                           0.87      6746
   macro avg       0.85      0.80      0.82      6746
weighted avg       0.86      0.87      0.86      6746


混淆矩阵:
[[4632  240]
 [ 665 1209]]


In [ ]:
# 待过滤数据设置
FILTER_BATCH_SIZE = 512
FILTER_THRESHOLD = 0.98
FILTER_PATH = "../data/baike_qa/baike_qa_train.json"

KEEP_OUTPUT_PATH = "../data/baike_qa_output.json"
DROPED_OUTPUT_PATH = "../data/baike_qa_filtered.json"

# 过滤新数据 
print("\n开始过滤新数据...")
test_sentences = []
fd = open(FILTER_PATH)
batch = []
for line in fd:
    info = json.loads(line)
    if len(batch) < FILTER_BATCH_SIZE:
        info["title"] = info["title"].replace("\n", "")
        info["answer"] = info["answer"].replace("\n", "")
        batch.append(info["title"] + "\n" + info["answer"])
    else:
        test_sentences.append(batch)
        batch = []
if batch:
    test_sentences.append(batch)

fw = open(KEEP_OUTPUT_PATH, "w")
fd = open(DROPED_OUTPUT_PATH, "w")
total_count, keep_count = 0, 0
for sentence in tqdm(test_sentences, total=len(test_sentences)):
    spam, confidence = predict_spam(model, tokenizer, sentence, device)
    total_count += 1
    for s, c, t in zip(spam, confidence, sentence): 
        title, answer = t.split("\n")
        json_str = {"title": title, "answer": answer}
        if s == 0 and c > FILTER_THRESHOLD:
            fw.write(json.dumps(json_str, ensure_ascii=False) + "\n")
            keep_count += 1
        else:
            fd.write(json.dumps(json_str, ensure_ascii=False) + "\n")
print(f"total doc num: {total_count}, keep doc num: {keep_count}")

In [21]:
! ls -al data/

total 1157364
drwxr-xr-x 4 root root      4096 Nov  9 16:20 .
drwxr-xr-x 6 root root      4096 Nov  9 17:43 ..
drwxr-xr-x 2 root root        61 Nov  9 15:01 .ipynb_checkpoints
drwxr-xr-x 3 root root       102 Nov  7 16:17 baike_qa
-rw-r--r-- 1 root root 180728797 Nov  9 14:18 baike_qa_filtered.json
-rw-r--r-- 1 root root    377527 Nov  9 17:43 baike_qa_filtered_tmp.json
-rw-r--r-- 1 root root 960441175 Nov  9 14:18 baike_qa_output.json
-rw-r--r-- 1 root root   1917085 Nov  9 17:43 baike_qa_output_tmp.json
-rw-r--r-- 1 root root   4479525 Nov  9 14:18 output_quality.json
-rw-r--r-- 1 root root  35855014 Nov  9 14:18 output_quality_full.json
-rw-r--r-- 1 root root   1127089 Nov  9 16:47 output_quality_full_tmp.json


In [22]:
! wc -l data/baike_qa_output.json

1076755 data/baike_qa_output.json


In [25]:
! wc -l ./data/baike_qa/baike_qa_train.json

1425170 ./data/baike_qa/baike_qa_train.json


In [26]:
! wc -l data/baike_qa_filtered.json

345637 data/baike_qa_filtered.json


In [16]:
! head -10 data/baike_qa_filtered.json

{"title": "黑河市治疗癫痫最权威的医院有哪些 ", "answer": "治疗癫痫病的时候是要很长的时间的，不是一两天就能够治好的，而且也不是一两个月就能够治好的。当然想要把癫痫病治好，那么癫痫患者就要去专业的医院治疗，只有专业的医院才能够把癫痫病治好。癫痫患者在找医院的时候要看看医院有没有先进的治疗设备，有没有很有效的治疗的方法，因为这些在治疗癫痫病的时候是很重要的。生了病就要赶紧的到医院治疗，当然是要到专业的医院，但是患者除了要找到正规的医院以外，患者也要找到治病的方法。治疗癫痫病的方法是有很多的，因为发作的原因不一样，所以在治疗的时候方法也就不一样了。因此癫痫患者在选择方法的时候要注意一些。\r治疗癫痫病不是很容易的事情，癫痫患者想要把癫痫病治好，那么癫痫患者就要在治疗的时候选对治疗的医院，这样癫痫患者的疾病才可以有不错的治疗的。癫痫患者在选择医院的时候要选择专业的医院，专业的医院有着权威的治疗专家的，也是有着好的治疗的方法的，这样癫痫患者的疾病就可以有好的治疗。治疗癫痫病的时候，癫痫患者要选择好的治疗的方法，这样癫痫患者的疾病才能得到好的治疗。癫痫患者在选择治疗方法的时候，患者先要把自己的病情确定清楚，然后患者在根据自己的病情来选择治疗的方法，这样癫痫患者的疾病才能得到好转的。"}
{"title": "本期铁胆罗马强势单3 ", "answer": "谢谢分享！！祝你多多中奖！！！"}
{"title": "邯郸市哪里有卖纳威沙发 ", "answer": "邯郸市亚森家具城内有卖的！ 地址：邯郸市中华大街天客隆超市对过即到！"}
{"title": "出大事了出大事了你知道吗武汉江夏公安分局接到群众举报电话,说在天 ", "answer": "好一个出大事了，好一个炸弹！\r又着实被你忽悠一下，还以为出什么大事了。"}
{"title": "下体很痒怎么回事?是什么原因呢?长了一些像疹子一样的东西,正好我 ", "answer": "恩，现在这种天气就容易出皮疹，没关系的，洗完澡在大腿根附近抹点痱子粉，天热容易出汗，那里潮湿所以就容易起你说的那种疹子。最好用带薄荷的卫生巾，像ABC卫生巾和护垫垫上凉凉的会舒服些！！"}
{"title": "重庆哪里可以治疗尖锐湿疣能不复发 ", "answer": "常规物理疗法，一般针对疣体较小、较少的情况

In [29]:
! head -10 data/baike_qa_output.json

{"title": "人站在地球上为什么没有头朝下的感觉 ", "answer": "地球上重力作用一直是指向球心的，因此\r只要头远离球心，人们就回感到头朝上。"}
{"title": "我的小baby", "answer": "勤洗澡，养成好的卫生习惯"}
{"title": "请问这起交通事故是谁的责任居多?小车和摩托车发生事故，在无红绿灯 ", "answer": "通过没有信号控制的十字路口，应该减速慢性，让右边的车先行，按你说的，摩托车好像在汽车的左边，所以严格来说可能摩托车全责。当然还要看汽车是否证照齐全，是否饮酒等。具体由交警调查后认定。"}
{"title": "松本面板可以配什么品牌的超五类模块?? ", "answer": "AMP的试试吧，还有普天的。"}
{"title": "请教怎么能很快能洗干净猪肠?有什么方法 ", "answer": "先用清水冲一下,在用食盐反复的搓揉,直到肠的黏液全去掉为止,在用清水反复冲洗干净,最好把肠上的油去掉一些再下锅。"}
{"title": "毛孔粗大怎么办我脸上长豆豆,出油,额头,鼻子上,脸上都有毛孔,很 ", "answer": "这是很多人都关心的问题。在此我们先关注毛孔收缩与舒张的问题。曾不止一次地看过一个关于洗脸的小常识，就是：在洗脸时应先用温热的水让毛孔打开，以便让毛孔里的脏污能够顺利洗去，洗完脸后再用冷水轻泼脸庞，有助于毛孔的收缩。毛孔这样一开一收的说法，虽然我们很难用肉眼观察到，不过，医生的回答却是至今还没有针对毛孔开与收做出任何具体的临床报告。 \r \r 然而，对于毛孔变大是否能缩小，却是有明确的数据证明。皮肤科医生指出，毛孔的确能缩小，关键就在于肌肤细胞间的保水度充足，让细胞间的空隙变小，进而往表皮层推挤后，毛孔自然就变小了。只是，要让一个毛孔原本很明显的人缩小到几乎看不见，就属于不可能的任务了。如果把肌肤用放大镜仔细端详，会发现年轻肌肤的纹理是呈米字形的，有彼此来回穿梭的线条以保住水分，所以肌肤较厚实，当然毛孔也就不明显。但当肌肤老化时，放大镜下的纹理就变成川字形了，平行的纹理是薄的、是无法保住水分的，所以毛孔也就松垮地变大了。 \r \r 两大关键毛孔是有弹性的，如果你可以减少毛孔的阻塞物，它自然就能缩小了。所以，要养成清理角质的习惯，不要等到粉刺有黑头现象才处

# 优化点

In [ ]:
# 1.数据层面，清洗里面的数据，把不准确的label纠正回来， K-fold cross-validation, [1,2,3,4,5], 5%+, 90+% (人工)
# 2.模型层面，调节正负比例，focal loss, label reweight, sample reweight, 针对不平衡分类来做，（训练技巧+魔改）
# 3.badcase闭环，短句预测不好，做数据增强，补充要正负样本都要涵盖到 (人工)
# ...